# 03 - Engenharia de features (Fase 3 - seleção de modelos)

**Regras que seguimos:**
1. **Uma feature herda a elegibilidade das colunas de onde vem**. Ex.: tudo o que usa `installment` (T2, *pricing*) só entra na classificação; tudo o que usa `issue_d` (T4) também.
2. **As features ficam dentro do pipeline**: o `FeatureBuilder` (`src/lending/feature_engineering.py`) é um transformer do scikit-learn e vem antes do `build_preprocessor`. Os cálculos são linha a linha e não aprendem nada com os dados, mas assim são aplicados sempre da mesma forma no treino, na validação e no teste.
3. **Divisões por zero e datas impossíveis dão NaN**. O pipeline trata esses NaN como os das outras colunas: mediana do treino de cada *fold* + indicador de "em falta".
4. Só dados de desenvolvimento. O teste final (2015) não é aberto.

**Divisão no grupo:** 5 features de cada elemento.

**Antes de correr:** é preciso ter corrido `01_preprocessamento.ipynb` (gera `regressao_dev.pkl` e `classificacao_treino.pkl`).

In [1]:
# Setup
import sys
from pathlib import Path

# Tenta importar o lending
try:
    import lending
except ModuleNotFoundError:
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / "src" / "lending").is_dir():
            sys.path.insert(0, str(parent / "src"))
            break
    import lending

import dataclasses

import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline

from lending import config
from lending.feature_engineering import (FEATURES, FeatureBuilder, add_engineered, catalog,
                                         check_leakage, input_columns, specs_for)
from lending.features import TASKS, TaskFeatures
from lending.preprocessing import ExpandingWindowSplit, build_preprocessor
from lending.storage import PickleStore

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

## 1. Fichas das features

Cada feature é um `FeatureSpec` que contem: colunas de origem, fórmula, motivo, tratamento dos valores em falta e momento a partir do qual existe (T0 pedido, T2 *pricing*, T4 concessão; ver `docs/01_formulacao_problema/momentos_previsao.md`).

In [2]:
catalog()

,autor,fontes,fórmula,momento,regression,classification,clustering,motivo,valores em falta
feature,,,,,,,,,
loan_to_income,João,"loan_amnt, annual_inc",loan_amnt / annual_inc,T0 Pedido,✓,✓,✓,"Peso do montante pedido face ao rendimento: o mesmo montante é um esforço diferente para rendimentos diferentes. Na EDA, o montante e o rendimento estão correlacionados (ρ = 0,50), mas nenhum dos dois, sozinho, mede o esforço.","NaN se annual_inc faltar ou for <= 0 (sem divisão por zero); depois, imputação pela mediana do treino + indicador no pipeline."
installment_to_income,João,"installment, annual_inc",100 × installment / (annual_inc / 12),T2 Pricing,✗,✓,✗,"Prestação mensal em % do rendimento mensal: mede o peso do empréstimo no orçamento, que nem a prestação nem o rendimento medem sozinhos (na EDA, a prestação em valor absoluto não tem relação com o default).",NaN se annual_inc faltar ou for <= 0; imputação + indicador no pipeline.
dti_with_loan,João,"dti, installment, annual_inc",dti + 100 × installment / (annual_inc / 12),T2 Pricing,✗,✓,✗,O dti da LendingClub exclui o empréstimo pedido; somar a nova prestação dá o endividamento do mutuário depois de o receber. Nota: é uma combinação linear de dti e installment_to_income; em modelos lineares e no kNN usa-se só uma das duas (decisão tomada na validação cruzada).,"NaN se dti, installment ou annual_inc faltarem; imputação + indicador."
credit_history_years,João,"issue_d, earliest_cr_line",(issue_d − earliest_cr_line) em meses / 12,T4 Concessão,✗,✓,✗,"Antiguidade do historial de crédito na data de concessão, como pede o enunciado para o classificador. Um historial curto dá menos informação sobre o comportamento do mutuário.",NaN se alguma das datas faltar ou se a diferença for negativa (data impossível); imputação + indicador.
issue_year,João,issue_d,ano de issue_d,T4 Concessão,✗,✓,✗,"Capta mudanças de política e de contexto económico ao longo do tempo (na EDA, a população de 2013–2014 é mais arriscada do que a de 2007–2011). O teste (2015) está fora dos anos de treino; como o pipeline limita cada variável aos percentis do treino, um ano posterior é tratado como o último ano do treino (a tendência não é extrapolada). A validação temporal, em que cada fold valida um ano que o treino não tem, mostra se a feature ajuda; a decisão de a manter é tomada aí.",Sem valores em falta (issue_d existe sempre).
revol_to_income,Rodrigo,"revol_bal, annual_inc",revol_bal / annual_inc,T0 Pedido,✓,✓,✓,"Saldo em crédito rotativo (cartões) face ao rendimento: o revol_util mede a parte do limite usada, mas não diz se a dívida é grande para o rendimento. No treino da classificação, o quintil mais alto tem 14,9% de default contra cerca de 11,3% nos três primeiros; a correlação com o revol_util (ρ = 0,34) e com o dti (ρ = 0,40) é moderada, por isso acrescenta informação.","NaN se revol_bal faltar ou se annual_inc faltar ou for <= 0 (sem divisão por zero); depois, imputação pela mediana do treino + indicador no pipeline."
open_acc_pct,Rodrigo,"open_acc, total_acc",100 × open_acc / total_acc,T0 Pedido,✓,✓,✓,"Parte do historial de crédito que continua aberta: o mesmo número de contas abertas significa coisas diferentes para quem teve 5 ou 50 contas. Uma percentagem alta indica crédito recente ou acumulado. No treino da classificação, o quintil mais alto tem 15,5% de default contra 10,5% a 12,1% nos outros (relação não linear); a correlação com o open_acc é só ρ = 0,26.","NaN se alguma das colunas faltar, se total_acc for <= 0 ou se open_acc for maior do que total_acc (caso impossível: 2 empréstimos de 2007); imputação pela mediana do treino + indicador no pipeline."
ever_delinquent,Rodrigo,"delinq_2yrs, mths_since_last_delinq",1 se delinq_2yrs > 0 ou mths_since_last_delinq preenchido; senão 0,T0 Pedido,✓,✓,✓,"Indica se o mutuário já teve algum atraso de pagamento. O delinq_2yrs só conta os últimos 2 anos e o mths_since_last_delinq vazio quer dizer ""nunca"", por isso juntar as duas colunas dá o historial completo 

## 2. Elegibilidade por tarefa e verificação contra *leakage*

| Tarefa | Momento de previsão | Que features entram |
|---|---|---|
| Regressão | T0 (antes do *pricing*) | só as que usam colunas de T0 |
| Classificação | T4 (concessão) | todas as que usam colunas até T4 |
| Clustering | perfil em T0 | só as de T0 |

O `check_leakage` usa as listas de colunas proibidas do enunciado (`features.py`). Se alguma feature de uma tarefa usar uma coluna proibida nessa tarefa dá erro. A última célula mostra o erro que aparece se alguém tentar usar `installment_to_income` na regressão.

In [3]:
for task in TASKS:
    check_leakage(task)
    print(f"{task:15s} {len(specs_for(task))} features: {[s.name for s in specs_for(task)]}")

regression      6 features: ['loan_to_income', 'revol_to_income', 'open_acc_pct', 'ever_delinquent', 'has_public_record', 'total_credit_util']
classification  9 features: ['loan_to_income', 'installment_to_income', 'dti_with_loan', 'credit_history_years', 'issue_year', 'revol_to_income', 'open_acc_pct', 'ever_delinquent', 'has_public_record']
clustering      5 features: ['loan_to_income', 'revol_to_income', 'open_acc_pct', 'ever_delinquent', 'has_public_record']


In [4]:
# Demonstração: a mesma feature, marcada (por engano) como elegível na regressão, é recusada
wrong = [dataclasses.replace(FEATURES[1], tasks=("regression", "classification"))]
try:
    check_leakage("regression", wrong)
except ValueError as error:
    print("Recusada:", error)

Recusada: A feature 'installment_to_income' usa colunas proibidas em regression: ['installment']


## 3. Criar as features nos dados de desenvolvimento

`input_columns` junta às colunas de entrada de cada tarefa as colunas de origem que só entram transformadas (na classificação, `issue_d` e `earliest_cr_line`). Depois do `FeatureBuilder` o `build_preprocessor` descarta-as.

In [5]:
regression = PickleStore.load(config.REG_DEV_PKL, expected_type=pd.DataFrame)
classification = PickleStore.load(config.CLS_TRAIN_PKL, expected_type=pd.DataFrame)
data = {"regression": regression, "classification": classification}

base = {task: TaskFeatures(task) for task in data}
built = {}
for task in data:
    builder = FeatureBuilder(task).fit(data[task][input_columns(base[task])])
    built[task] = builder.transform(data[task][input_columns(base[task])])
    print(f"{task}: {len(data[task]):,} linhas; features novas: {builder.names_}")

regression: 370,278 linhas; features novas: ['loan_to_income', 'revol_to_income', 'open_acc_pct', 'ever_delinquent', 'has_public_record', 'total_credit_util']
classification: 31,522 linhas; features novas: ['loan_to_income', 'installment_to_income', 'dti_with_loan', 'credit_history_years', 'issue_year', 'revol_to_income', 'open_acc_pct', 'ever_delinquent', 'has_public_record']


## 4. Verificações

- percentagem de valores em falta;
- nenhum infinito;
- distribuição de cada feature, para confirmar que os valores fazem sentido;
- regras de coerência: o DTI com o novo empréstimo nunca é menor do que o DTI; a antiguidade do crédito nunca é negativa; o ano está dentro do período de treino.
- features do Rodrigo: os rácios nunca são negativos; a percentagem de contas abertas fica entre 0 e 100; as duas flags só têm 0, 1 ou NaN; a utilização total do crédito só existe na regressão e só falta quando o limite total falta ou é 0.

In [6]:
for task, X in built.items():
    names = [s.name for s in specs_for(task)]
    table = X[names].describe(percentiles=[0.01, 0.5, 0.99]).T
    table.insert(1, "falta_%", X[names].isna().mean().mul(100))
    table.insert(2, "infinitos", np.isinf(X[names]).sum())
    print(task)
    display(table.round(2))

regression


,count,falta_%,infinitos,mean,std,min,1%,50%,99%,max
loan_to_income,370278.0,0.0,0,0.22,0.11,0.0,0.03,0.21,0.45,0.50
revol_to_income,370278.0,0.0,0,0.24,0.19,0.0,0.01,0.20,0.86,7.96
open_acc_pct,370278.0,0.0,0,48.51,17.00,0.0,17.14,46.67,94.74,100.00
ever_delinquent,370278.0,0.0,0,0.48,0.50,0.0,0.00,0.00,1.00,1.00
has_public_record,370278.0,0.0,0,0.16,0.36,0.0,0.00,0.00,1.00,1.00
total_credit_util,370263.0,0.0,0,74.25,18.86,0.0,20.07,79.16,102.91,331.18


classification


,count,falta_%,infinitos,mean,std,min,1%,50%,99%,max
loan_to_income,31518.0,0.01,0,0.17,0.11,0.00,0.02,0.15,0.48,1.34
installment_to_income,31518.0,0.01,0,6.52,4.15,0.03,0.76,5.63,19.05,54.17
dti_with_loan,31518.0,0.01,0,19.58,8.28,0.04,3.11,19.50,37.82,69.17
credit_history_years,31493.0,0.09,0,13.31,6.91,0.50,3.08,12.08,35.92,65.42
issue_year,31522.0,0.00,0,2010.07,1.04,2007.00,2007.00,2010.00,2011.00,2011.00
revol_to_income,31518.0,0.01,0,0.21,0.22,0.00,0.00,0.15,1.00,4.29
open_acc_pct,31491.0,0.10,0,48.38,19.30,5.13,15.15,45.45,100.00,100.00
ever_delinquent,31493.0,0.09,0,0.35,0.48,0.00,0.00,0.00,1.00,1.00
has_public_record,31493.0,0.09,0,0.05,0.23,0.00,0.00,0.00,1.00,1.00


In [7]:
X = built["classification"]
both = X[["dti", "dti_with_loan"]].dropna()
assert (both["dti_with_loan"] >= both["dti"]).all()
assert (X["credit_history_years"].dropna() >= 0).all()
assert X["issue_year"].between(2007, 2011).all()
assert not np.isinf(X[[s.name for s in specs_for("classification")]]).any().any()
print("Regras de coerência verificadas.")

Regras de coerência verificadas.


In [8]:
# Coerência das features do Rodrigo (regressão e classificação)
for task, X in built.items():
    assert (X["revol_to_income"].dropna() >= 0).all()
    assert X["open_acc_pct"].dropna().between(0, 100).all()
    for flag in ["ever_delinquent", "has_public_record"]:
        assert set(X[flag].dropna().unique()) <= {0.0, 1.0}
    # A flag de atrasos só é NaN quando as duas colunas de origem faltam
    both_missing = X["delinq_2yrs"].isna() & X["mths_since_last_delinq"].isna()
    assert (X["ever_delinquent"].isna() == both_missing).all()

X = built["regression"]
util = X["total_credit_util"]
assert (util.dropna() >= 0).all()
no_limit = X["tot_hi_cred_lim"].isna() | (X["tot_hi_cred_lim"] <= 0) | X["tot_cur_bal"].isna()
assert (util.isna() == no_limit).all()
assert "total_credit_util" not in built["classification"].columns   # só na regressão
print(f"total_credit_util: {util.isna().sum()} em falta (limite em falta ou 0); "
      f"{(util > 100).sum():,} acima de 100% (saldo acima do limite, ficam)")
print("Regras de coerência das features do Rodrigo verificadas.")

total_credit_util: 15 em falta (limite em falta ou 0); 5,536 acima de 100% (saldo acima do limite, ficam)
Regras de coerência das features do Rodrigo verificadas.


## 5. Relação com o alvo (só treino, descritivo)

Para confirmar que as features têm significado, compara-se cada uma com o alvo **nos dados de treino**. É só descrição, pois a escolha final das features é feita dentro da validação cruzada.

In [9]:
# Regressão: correlação de Spearman com int_rate
X = built["regression"]
pd.Series({name: X[name].corr(regression["int_rate"], method="spearman")   # mesmo índice
           for name in [s.name for s in specs_for("regression")]}, name="rho_int_rate").round(3)

loan_to_income       0.200
revol_to_income      0.042
open_acc_pct         0.081
ever_delinquent      0.079
has_public_record    0.067
total_credit_util    0.109
Name: rho_int_rate, dtype: float64

In [10]:
# Classificação: correlação com o default e taxa de default (%) por quintil de cada feature
X = built["classification"]
is_default = (classification[config.CLASS_COL] == "Default").astype(int)   # mesmo índice que X
names = [s.name for s in specs_for("classification")]
display(pd.Series({n: X[n].corr(is_default, method="spearman") for n in names},
                  name="rho_default").round(3))

quintiles = {}
for n in names:
    if X[n].nunique() <= 2:   # flags 0/1: ver a célula seguinte
        continue
    rates = is_default.groupby(pd.qcut(X[n], q=5, duplicates="drop"), observed=True).mean()
    quintiles[n] = pd.Series(rates.mul(100).round(1).to_numpy(),
                             index=[f"Q{i}" for i in range(1, len(rates) + 1)])
pd.DataFrame(quintiles).T.rename_axis("% default por quintil")

loan_to_income           0.043
installment_to_income    0.053
dti_with_loan            0.063
credit_history_years    -0.042
issue_year              -0.074
revol_to_income          0.035
open_acc_pct             0.039
ever_delinquent          0.026
has_public_record        0.056
Name: rho_default, dtype: float64

,Q1,Q2,Q3,Q4,Q5
% default por quintil,,,,,
loan_to_income,11.2,11.0,11.8,12.0,15.4
installment_to_income,10.9,11.1,11.3,12.4,15.9
dti_with_loan,9.3,10.7,12.7,13.9,14.8
credit_history_years,14.5,12.7,12.2,11.5,10.6
issue_year,16.6,10.9,10.6,NaN,NaN
revol_to_income,11.4,11.2,11.5,12.5,14.9
open_acc_pct,11.9,10.5,11.5,12.1,15.5


In [11]:
# Flags (0/1): taxa de default (%) e número de empréstimos para cada valor
flags = [n for n in names if X[n].nunique() <= 2]
pd.concat({n: pd.DataFrame({"empréstimos": X[n].value_counts().sort_index(),
                            "% default": is_default.groupby(X[n]).mean().mul(100).round(1)})
           for n in flags}).rename_axis(["flag", "valor"])

empréstimos  % default
flag              valor                        
ever_delinquent   0.0          20532       11.7
                  1.0          10961       13.5
has_public_record 0.0          29765       11.9
                  1.0           1728       19.9

## 6. As features no pipeline (*fit* só no treino de cada *fold*)

O pipeline completo é `FeatureBuilder` -> `build_preprocessor`. Em cada *fold* da validação temporal é ajustado de novo, só com o treino desse *fold*, e aplicado à validação. As features novas passam pelos mesmos passos das outras numéricas, *clipping* aos percentis do treino, imputação pela mediana + indicador e padronização.

In [12]:
splitters = {"regression": ExpandingWindowSplit("Q", min_train_periods=2),
             "classification": ExpandingWindowSplit("Y", min_train_periods=2)}
rows = []
for task, splitter in splitters.items():
    cols, extended = input_columns(base[task]), add_engineered(base[task])
    for fold, (train_idx, valid_idx) in enumerate(splitter.split(data[task]["issue_d"]), start=1):
        pipe = Pipeline([("features", FeatureBuilder(task)), ("pre", build_preprocessor(extended))])
        X_train = pipe.fit_transform(data[task].iloc[train_idx][cols])
        X_valid = pipe.transform(data[task].iloc[valid_idx][cols])
        assert X_train.notna().all().all() and X_valid.notna().all().all()
        new = [n for n in pipe["features"].names_ if n in X_train.columns]
        rows.append({"tarefa": task, "fold": fold, "linhas treino": len(train_idx),
                     "linhas validação": len(valid_idx), "colunas de saída": X_train.shape[1],
                     "features novas na saída": len(new)})
pd.DataFrame(rows).set_index(["tarefa", "fold"])

linhas treino  linhas validação  colunas de saída  \
tarefa         fold                                                      
regression     1             53350             37562                97   
               2             90912             43855                97   
               3            134767             47388                98   
               4            182155             55332                97   
               5            237487             58693                97   
               6            296180             74098                99   
classification 1              2991              5280               123   
               2              8271              9153               120   
               3             17424             14098               117   

                     features novas na saída  
tarefa         fold                           
regression     1                           6  
               2                           6  
               3                           6  
               4                           6  
               5                           6  
               6                           6  
classification 1                           9  
               2                           9  
               3                           9

## 7. Resumo

| O quê | Onde |
|---|---|
| Fichas das features | `catalog()`, `docs/03_features/features.md`, secção 3 do relatório |
| Cálculo | `FeatureBuilder` (`src/lending/feature_engineering.py`) |
| Verificação contra *leakage* | `check_leakage` (listas do enunciado em `features.py`) |
| Entrada no pipeline | `add_engineered` + `input_columns` + `build_preprocessor` |

**Como se usa na Parte 2:**

```python
base = TaskFeatures("classification")
pipe = Pipeline([("features", FeatureBuilder("classification")),
                 ("pre", build_preprocessor(add_engineered(base))),
                 ("model", modelo)])
pipe.fit(treino[input_columns(base)], y_treino)
```